In [74]:

from hashlib import md5


In [75]:


def add_experiment(experiment, buckets):
    """
    checks if it is possible to add an experiment and adds it
    experiments are distributed in such a way to fit as many experiments as possible

    :param experiment (dict):
        - parameters of the experiment, that should be added
        - Dictionary keys:
            - id - experiment id
            - buckets_count - necessary number of buckets
            - conflicts - list of experiment ids, that cannot be simultaneously run on the same users

    :param buckets (list[list[int]]):
            - list of buckets
            - each bucket is a list
            - each bucket contains experiment ids, that are running inside this bucket

    :return (success, buckets):
        success (boolean) - True (if it is possible to add an experiment), False (otherwise)
        buckets (list[list[int]]) - updated buckets (if it is possible to add an experiment)
    """

    # check that there are enough buckets
    if experiment['buckets_count'] > len(buckets):
        return False, buckets

    # check buckets for conflicts and find indexes of available buckets
    available_indexes = []
    for idx, bucket in enumerate(buckets):

        found_conflict = False
        for conflict in experiment['conflicts']:
            if conflict in bucket:
                found_conflict = True
                break

        if not found_conflict:
            available_indexes.append(idx)

    # check that there are enough available buckets without conflict
    if experiment['buckets_count'] > len(available_indexes):
        return False, buckets

    # add experiment to available buckets
    for i in range(experiment['buckets_count']):
        buckets[available_indexes[i]].append(experiment['id'])

    return True, buckets


In [81]:

# generate initial empty buckets
total_buckets_count = 4
buckets = [[] for _ in range(total_buckets_count)]

# TEST CASE 01: more buckets are needed for the experiment than is available
# success, buckets = False, [[], [], [], []]
success, buckets = add_experiment({'id': 0, 'buckets_count': 5, 'conflicts': []}, buckets)
if success == False and buckets == [[], [], [], []]:
    print('test_case_01: passed')
else:
    print('test_case_01: failed')

# TEST CASE 02:
success, buckets = add_experiment({'id': 1, 'buckets_count': 4, 'conflicts': [4]}, buckets)
if success == True and buckets == [[1], [1], [1], [1]]:
    print('test_case_02: passed')
else:
    print('test_case_02: failed')

# TEST CASE 03:
# experiment with id=2 can be in any 2 out of 4 buckets
success, buckets = add_experiment({'id': 2, 'buckets_count': 2, 'conflicts': [3]}, buckets)
if success == True and buckets == [[1, 2], [1, 2], [1], [1]]:
    print('test_case_03: passed')
else:
    print('test_case_03: failed')

# TEST CASE 04:
# possible to add to buckets where experiment with id=2 is not running
success, buckets = add_experiment({'id': 3, 'buckets_count': 2, 'conflicts': [2]}, buckets)
if success == True and buckets == [[1, 2], [1, 2], [1, 3], [1, 3]]:
    print('test_case_04: passed')
else:
    print('test_case_04: failed')

# TEST CASE 05:
# not possible to add, since in every bucket there is a conflict with experiment-1
success, buckets = add_experiment({'id': 4, 'buckets_count': 1, 'conflicts': [1]}, buckets)
if success == False and buckets == [[1, 2], [1, 2], [1, 3], [1, 3]]:
    print('test_case_05: passed')
else:
    print('test_case_05: failed')


test_case_01: passed
test_case_02: passed
test_case_03: passed
test_case_04: passed
test_case_05: passed


In [77]:

def get_group(value: str, n_groups: int, salt: str= ''):
    """
    Get experimental bucket or group based on id
    :param value: unique id of an object
    :param n_groups: number of buckets or other groups
    :param salt for hashing
    :return: group id (from 0 to n)
    """
    hashed_id = int(md5((value + salt).encode()).hexdigest(), 16)
    group_id = hashed_id % n_groups
    return group_id

n_users = 100
n_buckets = 10
buckets_01 = [get_group(str(value), n_buckets, 'salt_one') for value in range(n_users)]
buckets_02 = [get_group(str(value), n_buckets, 'salt_one') for value in range(n_users)]
buckets_03 = [get_group(str(value), n_buckets, 'salt_two') for value in range(n_users)]
print(buckets_01 == buckets_02)
print(buckets_02 == buckets_03)


True
False


In [78]:

def process_user(user_id: str, buckets: list, experiments: list, bucket_salt: str):
    """
    Assign user to experiments:
    1. assign user to a bucket
    2. for each experiment in the bucket select control or pilot group
    Затем для каждого эксперимента в этом бакете выбрать пилотную или контрольную группу

    :param user_id: user id

    :param buckets:
        - list of buckets (list[list[int]])
        - each bucket is a list of int
        - each bucket contains experiment ids, that are running inside this bucket

    :param experiments:
        - list of dictionaries with information about experiments (list[dict])
        - dictionary keys:
            - id - experiment id
            - salt (str) - salt for experiment to distribute users into control and pilot groups

    :param bucket_salt:
        - salt to distribute users into buckets
        - the same salt should distribute users into the same buckets
        - if this salt is changed => distribution of users into buckets should also change

    :return bucket_id, experiment_groups:
        - bucket_id (int) - bucket id (index of the bucket in the 'buckets' list)
        - experiment_groups (list[tuple])
            - list of pairs: experiment id (int), group ('A' or 'B')
            - example: (8, [(194, 'A'), (73, 'B')])
    """

    # it seems that I need to do 3 things:
    # - assign bucket
    # - assign experiment
    # - assign group
    # it feels like I need to only:
    # - assign bucket
    # - assign group

    # assign bucket to a user
    bucket_id = get_group(value=user_id, n_groups=len(buckets), salt=bucket_salt)
    experiment_groups = []

    # assign group (A or B) to a user
    bucket = buckets[bucket_id]
    for experiment_id in bucket:
        # find experiment by id
        experiment = next((e for e in experiments if e['id'] == experiment_id), None)

        # assign group in the experiment to the user (control or experimental)
        group_id = get_group(value=user_id, n_groups=2, salt=experiment['salt'])
        experiment_groups.append((experiment['id'], 'A' if group_id == 0 else 'B'))

    return bucket_id, experiment_groups


In [79]:

user_id = '1001'
experiments = [{'id': 0, 'salt': '0'}, {'id': 1, 'salt': '1'}]
buckets = [[0, 1], [1], []]
bucket_salt = 'a2N4'
bucket_id, experiment_groups = process_user(user_id, buckets, experiments, bucket_salt)
print(f'bucket_id = {bucket_id}, experiment_groups = {experiment_groups}')
# В зависимости от значений bucket_salt и солей экспериментов, можно получить один из вариантов:
# bucket_id, experiment_groups = 0, [(0, 'A'), (1, 'A')]
# bucket_id, experiment_groups = 0, [(0, 'A'), (1, 'B')]
# bucket_id, experiment_groups = 0, [(0, 'B'), (1, 'A')]
# bucket_id, experiment_groups = 0, [(0, 'B'), (1, 'B')]
# bucket_id, experiment_groups = 1, [(1, 'A')]
# bucket_id, experiment_groups = 1, [(1, 'B')]
# bucket_id, experiment_groups = 2, []


bucket_id = 0, experiment_groups = [(0, 'A'), (1, 'A')]
